Read silver table

In [0]:
from pyspark.sql import functions as F

employee_silver_df = spark.table(
    "databricks_project1.silver.employee_payroll"
)

print("Silver Employee table loaded successfully")
print(f"Silver Employee row count: {employee_silver_df.count()}")

employee_silver_df.printSchema()

Select and standardize employee_data

In [0]:
# ---------------------------------------------------------
# Select and standardize employee data
# ---------------------------------------------------------

employee_base_df = (
    employee_silver_df
    .select(
        F.col("Employee_Code").cast("string").alias("Employee_Code"),
        F.col("Badge_#").cast("int").alias("Badge_#"),
        F.trim(F.col("Employee_Status")).alias("Employee_Status"),
        F.trim(F.col("FirstName")).alias("First_Name"),
        F.trim(F.col("LastName")).alias("Last_Name"),
        F.col("Facility_Code").cast("int").alias("Facility_Code"),
        F.trim(F.col("facname")).alias("Facility_Name"),
        F.trim(F.col("Incharge")).alias("Incharge"),
        F.col("Labor_Position_Code").cast("int").alias(
            "Labor_Position_Code"
        ),
        F.col("DOB").alias("Birth_Date"),
        F.col("Employee_Added").alias("Employee_Added"),
        F.col("Hire_Date").alias("Hire_Date"),
        F.col("Rehire_Date").alias("Rehire_Date"),
        F.col("Termination_Date").alias("Termination_Date"),
        F.col("ingestion_timestamp").alias("ingestion_timestamp"),
        F.col("source_file").alias("source_file")
    )
)

print(
    "Base Employee records:",
    employee_base_df.count()
)

Create the latest Silver Employee DataFrame

In [0]:
# ---------------------------------------------------------
# Prepare latest Silver Employee records
# One latest record per Employee_Code
# ---------------------------------------------------------

from pyspark.sql.window import Window

employee_window = (
    Window
    .partitionBy("Employee_Code")
    .orderBy(
        F.to_timestamp("ingestion_timestamp").desc()
    )
)

employee_latest_df = (
    employee_base_df
    .withColumn(
        "_rn",
        F.row_number().over(employee_window)
    )
    .filter(
        F.col("_rn") == 1
    )
    .drop("_rn")
)

print(
    "Silver Employee records:",
    employee_base_df.count()
)

print(
    "Latest unique Employee records:",
    employee_latest_df.count()
)

print(
    "Distinct Employee_Code:",
    employee_latest_df
    .select("Employee_Code")
    .distinct()
    .count()
)

Read the current DimEmployee records

In [0]:
# ---------------------------------------------------------
# Read current DimEmployee records
# ---------------------------------------------------------

dim_employee_gold_df = (
    spark.table(
        "databricks_project1.gold.dim_employee"
    )
    .filter(
        F.col("IsCurrent") == True
    )
)

print(
    "Current DimEmployee records:",
    dim_employee_gold_df.count()
)

print(
    "Distinct current Employee_Code:",
    dim_employee_gold_df
    .select("Employee_Code")
    .distinct()
    .count()
)

Detect NEW employees

In [0]:
# ---------------------------------------------------------
# Identify new employees
# ---------------------------------------------------------

new_employee_df = (
    employee_latest_df.alias("src")
    .join(
        dim_employee_gold_df
        .select("Employee_Code")
        .alias("tgt"),
        F.col("src.Employee_Code") ==
        F.col("tgt.Employee_Code"),
        "left_anti"
    )
)

print(
    "New Employee records:",
    new_employee_df.count()
)

Detect CHANGED employees

In [0]:
# ---------------------------------------------------------
# Identify changed employees
# Null-safe comparison
# ---------------------------------------------------------

changed_employee_df = (
    employee_latest_df.alias("src")
    .join(
        dim_employee_gold_df.alias("tgt"),
        F.col("src.Employee_Code") ==
        F.col("tgt.Employee_Code"),
        "inner"
    )
    .filter(
        ~F.col("src.Employee_Status")
        .eqNullSafe(F.col("tgt.Employee_Status"))
        |
        ~F.col("src.First_Name")
        .eqNullSafe(F.col("tgt.First_Name"))
        |
        ~F.col("src.Last_Name")
        .eqNullSafe(F.col("tgt.Last_Name"))
        |
        ~F.col("src.Facility_Code")
        .eqNullSafe(F.col("tgt.Facility_Code"))
        |
        ~F.col("src.Labor_Position_Code")
        .eqNullSafe(F.col("tgt.Labor_Position_Code"))
        |
        ~F.col("src.Birth_Date")
        .eqNullSafe(F.col("tgt.Birth_Date"))
        |
        ~F.col("src.Hire_Date")
        .eqNullSafe(F.col("tgt.Hire_Date"))
        |
        ~F.col("src.Rehire_Date")
        .eqNullSafe(F.col("tgt.Rehire_Date"))
        |
        ~F.col("src.Termination_Date")
        .eqNullSafe(F.col("tgt.Termination_Date"))
    )
)

print(
    "Changed Employee records:",
    changed_employee_df.count()
)

In [0]:
# ---------------------------------------------------------
# Prepare records requiring a new SCD version
# ---------------------------------------------------------

source_columns = employee_latest_df.columns

changed_source_df = (
    changed_employee_df
    .select(
        *[
            F.col(f"src.{c}").alias(c)
            for c in source_columns
        ]
    )
)

scd_insert_df = (
    new_employee_df
    .select(*source_columns)
    .unionByName(
        changed_source_df
    )
)

print(
    "SCD records to insert:",
    scd_insert_df.count()
)

In [0]:
# ---------------------------------------------------------
# SCD Type 2 - Write changes to DimEmployee
# ---------------------------------------------------------

from delta.tables import DeltaTable
from pyspark.sql import functions as F

DIM_EMPLOYEE_TABLE = "databricks_project1.gold.dim_employee"

if scd_insert_df.count() == 0:

    print("No new or changed Employee records to load.")
    print("DimEmployee remains unchanged.")

else:

    # -----------------------------------------------------
    # Step 1: Expire existing current records
    # for changed employees
    # -----------------------------------------------------

    changed_codes_df = (
        changed_employee_df
        .select(
            F.col("src.Employee_Code")
            .alias("Employee_Code")
        )
        .distinct()
    )

    dim_employee_delta = DeltaTable.forName(
        spark,
        DIM_EMPLOYEE_TABLE
    )

    (
        dim_employee_delta.alias("tgt")
        .merge(
            changed_codes_df.alias("src"),
            """
            tgt.Employee_Code = src.Employee_Code
            AND tgt.IsCurrent = true
            """
        )
        .whenMatchedUpdate(
            set={
                "IsCurrent": "false",
                "EndDate": "current_date()"
            }
        )
        .execute()
    )

    print(
        "Expired changed Employee records:",
        changed_codes_df.count()
    )

    # -----------------------------------------------------
    # Step 2: Prepare new current versions
    # -----------------------------------------------------

    new_scd_rows_df = (
        scd_insert_df
        .withColumn(
            "EmployeeKey",
            F.monotonically_increasing_id()
        )
        .withColumn(
            "StartDate",
            F.current_date()
        )
        .withColumn(
            "EndDate",
            F.to_date(F.lit("9999-12-31"))
        )
        .withColumn(
            "IsCurrent",
            F.lit(True)
        )
        .select(
            "EmployeeKey",
            "Employee_Code",
            "Badge_#",
            "Employee_Status",
            "First_Name",
            "Last_Name",
            "Facility_Code",
            "Facility_Name",
            "Incharge",
            "Labor_Position_Code",
            "Birth_Date",
            "Employee_Added",
            "Hire_Date",
            "Rehire_Date",
            "Termination_Date",
            "StartDate",
            "EndDate",
            "IsCurrent",
            "ingestion_timestamp",
            "source_file"
        )
    )

    # -----------------------------------------------------
    # Step 3: Insert new current versions
    # -----------------------------------------------------

    (
        new_scd_rows_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(DIM_EMPLOYEE_TABLE)
    )

    print(
        "New SCD Employee records inserted:",
        new_scd_rows_df.count()
    )

In [0]:
# ---------------------------------------------------------
# Validate DimEmployee after SCD processing
# ---------------------------------------------------------

gold_employee_df = spark.table(
    DIM_EMPLOYEE_TABLE
)

current_employee_df = (
    gold_employee_df
    .filter(F.col("IsCurrent") == True)
)

print(
    "Total DimEmployee rows:",
    gold_employee_df.count()
)

print(
    "Distinct Employee_Code:",
    gold_employee_df
    .select("Employee_Code")
    .distinct()
    .count()
)

print(
    "Current Employee records:",
    current_employee_df.count()
)

duplicate_current_df = (
    current_employee_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate current Employee_Code:",
    duplicate_current_df.count()
)

Check duplicate business keys

In [0]:
duplicate_employee_df = (
    employee_base_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicate Employee_Code count: "
    f"{duplicate_employee_df.count()}"
)

display(duplicate_employee_df)

Create the SCD Type 2 Gold DataFrame

In [0]:
%skip
# ---------------------------------------------------------
# Create Initial SCD Type 2 Gold DataFrame
# One current record per Employee_Code
# ---------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

current_date = F.current_date()

# Keep latest Silver record for each Employee_Code
employee_window = (
    Window
    .partitionBy("Employee_Code")
    .orderBy(
        F.to_timestamp("ingestion_timestamp").desc()
    )
)

employee_latest_df = (
    employee_base_df
    .withColumn(
        "_rn",
        F.row_number().over(employee_window)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

final_employee_df = (
    employee_latest_df

    .withColumn(
        "EmployeeKey",
        F.monotonically_increasing_id()
    )

    .withColumn(
        "StartDate",
        current_date
    )

    .withColumn(
        "EndDate",
        F.to_date(F.lit("9999-12-31"))
    )

    .withColumn(
        "IsCurrent",
        F.lit(True)
    )

    .select(
        "EmployeeKey",
        "Employee_Code",
        "Badge_#",
        "Employee_Status",
        "First_Name",
        "Last_Name",
        "Facility_Code",
        "Facility_Name",
        "Incharge",
        "Labor_Position_Code",
        "Birth_Date",
        "Employee_Added",
        "Hire_Date",
        "Rehire_Date",
        "Termination_Date",
        "StartDate",
        "EndDate",
        "IsCurrent",
        "ingestion_timestamp",
        "source_file"
    )
)

print(
    "Initial SCD Employee count:",
    final_employee_df.count()
)

print(
    "Distinct Employee_Code:",
    final_employee_df
    .select("Employee_Code")
    .distinct()
    .count()
)

print(
    "Current records:",
    final_employee_df
    .filter(F.col("IsCurrent") == True)
    .count()
)

create the gold table

In [0]:
%skip
# ---------------------------------------------------------
# Initial clean Gold DimEmployee load
# ---------------------------------------------------------

(
    final_employee_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "databricks_project1.gold.dim_employee"
    )
)

print(
    "Gold DimEmployee initial SCD load completed successfully."
)

verify the table

In [0]:
# ---------------------------------------------------------
# Validate Gold DimEmployee
# ---------------------------------------------------------

gold_employee_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

print(
    "Gold DimEmployee count:",
    gold_employee_df.count()
)

print(
    "Distinct Employee_Code:",
    gold_employee_df
    .select("Employee_Code")
    .distinct()
    .count()
)

current_record_count = (
    gold_employee_df
    .filter(F.col("IsCurrent") == True)
    .count()
)

print(
    "Current employee records:",
    current_record_count
)

duplicate_current_df = (
    gold_employee_df
    .filter(F.col("IsCurrent") == True)
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate current Employee_Code:",
    duplicate_current_df.count()
)

display(duplicate_current_df)

SCD Validation

In [0]:
invalid_scd_df = gold_employee_df.filter(
    (F.col("StartDate").isNull()) |
    (F.col("EndDate").isNull()) |
    (F.col("IsCurrent").isNull())
)

print(
    f"Invalid SCD records: {invalid_scd_df.count()}"
)

Audit-column validation

In [0]:
audit_null_count = gold_employee_df.filter(
    F.col("ingestion_timestamp").isNull() |
    F.col("source_file").isNull()
).count()

print(f"Rows with NULL audit columns: {audit_null_count}")

In [0]:
gold_employee_df.groupBy("IsCurrent").count().show()

In [0]:
gold_employee_df.printSchema()

In [0]:
%skip
#Temp cell

# ---------------------------------------------------------
# Inspect current DimEmployee state
# ---------------------------------------------------------
from pyspark.sql import functions as F
dim_employee_check_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

print(
    "Total DimEmployee rows:",
    dim_employee_check_df.count()
)

print(
    "Distinct Employee_Code:",
    dim_employee_check_df
    .select("Employee_Code")
    .distinct()
    .count()
)

print(
    "Current IsCurrent=True rows:",
    dim_employee_check_df
    .filter(F.col("IsCurrent") == True)
    .count()
)

duplicate_current_df = (
    dim_employee_check_df
    .filter(F.col("IsCurrent") == True)
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Employee_Code with multiple current records:",
    duplicate_current_df.count()
)

display(duplicate_current_df)

In [0]:
%skip
#Temp validation cells
# ---------------------------------------------------------
# Validate current DimEmployee uniqueness before SCD write
# ---------------------------------------------------------

current_dim_check_df = (
    spark.table(
        "databricks_project1.gold.dim_employee"
    # )
    .filter(
        F.col("IsCurrent") == True
    )
)

print(
    "Current DimEmployee rows:",
    current_dim_check_df.count()
)

print(
    "Distinct Employee_Code:",
    current_dim_check_df
    .select("Employee_Code")
    .distinct()
    .count()
)

duplicate_current_check_df = (
    current_dim_check_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate current Employee_Code:",
    duplicate_current_check_df.count()
)